# IEEE-CIS Fraud Detection — Exploratory Data Analysis

**Purpose:** understand the data before building the model. This notebook is exploration only — no production code lives here.

**What we answer:**
1. How imbalanced is the dataset? (motivates PR-AUC over accuracy)
2. Where are the missing values? (validates our fill strategy)
3. Do fraud transactions look different in amount and time?
4. How many unique values do key categoricals have? (motivates `category` dtype)

Run from the repo root: `poetry run jupyter notebook notebooks/01_eda.ipynb`

In [ ]:
from pathlib import Path
import sys

# Resolve repo root from the notebook file's absolute path.
# VS Code injects __vsc_ipynb_file__; classic Jupyter falls back to Path('..').
try:
    REPO_ROOT = Path(__vsc_ipynb_file__).parent.parent  # type: ignore[name-defined]
except NameError:
    REPO_ROOT = Path("..").resolve()

sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from fraud_detection.config import load_config
from fraud_detection.data.loader import load_raw
from fraud_detection.features.pipeline import build_features, extract_target

In [ ]:
cfg = load_config(REPO_ROOT / "config/config.yaml")

df, stats = load_raw(
    transaction_path=REPO_ROOT / cfg.data.transaction_path,
    identity_path=REPO_ROOT / cfg.data.identity_path,
)

print(f"Rows: {stats.n_rows:,}")
print(f"Fraud transactions: {stats.n_fraud:,} ({stats.fraud_rate:.2%})")
print(f"Overall missing rate: {stats.missing_rate:.2%}")
print(f"Columns: {df.shape[1]}")

## 1. Class imbalance

This is why accuracy is useless here. A model that always predicts 'not fraud' is ~96.5% accurate.

In [ ]:
counts = df['isFraud'].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Legit (0)', 'Fraud (1)'], counts.values, color=['steelblue', 'crimson'])
ax.set_title('Class distribution')
ax.set_ylabel('Transaction count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 1000, f'{v:,}\n({v/len(df):.1%})', ha='center')
plt.tight_layout()
plt.show()

## 2. Missing values

Top 30 columns by missing rate. This validates our fill strategy: sentinel -999 for numerics, 'missing' for categoricals.

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False).head(30)
fig, ax = plt.subplots(figsize=(10, 6))
missing.plot.barh(ax=ax, color='salmon')
ax.set_xlabel('Missing rate')
ax.set_title('Top 30 columns by missing rate')
ax.axvline(0.5, color='red', linestyle='--', label='50% threshold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Columns with >50% missing: {(df.isnull().mean() > 0.5).sum()}')

## 3. Transaction amount by fraud label

Do fraud transactions cluster at different amounts?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, ax in zip([0, 1], ['steelblue', 'crimson'], axes):
    subset = df[df['isFraud'] == label]['TransactionAmt']
    subset.clip(upper=subset.quantile(0.99)).plot.hist(
        bins=50, ax=ax, color=color, alpha=0.8
    )
    ax.set_title(f"TransactionAmt — {'Fraud' if label else 'Legit'}")
    ax.set_xlabel('Amount (clipped at 99th pct)')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

print(df.groupby('isFraud')['TransactionAmt'].describe().round(2))

## 4. Time pattern of fraud (TransactionDT)

TransactionDT is seconds from a reference point — not a real timestamp, but the relative pattern is still meaningful.

In [ ]:
# Convert to hours-of-day proxy (seconds mod 86400 / 3600)
df['hour'] = (df['TransactionDT'] % 86400) / 3600

fig, ax = plt.subplots(figsize=(10, 4))
for label, color, alpha in [(0, 'steelblue', 0.5), (1, 'crimson', 0.8)]:
    df[df['isFraud'] == label]['hour'].plot.hist(
        bins=24, ax=ax, color=color, alpha=alpha,
        label=f"{'Fraud' if label else 'Legit'}", density=True
    )
ax.set_title('Hour-of-day distribution (normalised)')
ax.set_xlabel('Hour of day')
ax.legend()
plt.tight_layout()
plt.show()

df.drop(columns=['hour'], inplace=True)  # clean up the temp column

## 5. Categorical cardinality

High-cardinality categoricals benefit from LightGBM's native `category` dtype handling over one-hot encoding.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
cardinality = df[cat_cols].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
cardinality.plot.bar(ax=ax, color='mediumpurple')
ax.set_title('Unique values per categorical column')
ax.set_ylabel('Cardinality')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(cardinality.to_string())

## 6. Feature pipeline sanity check

Run `build_features()` on the full DataFrame — confirm output shape, no NaNs, category dtypes present.

In [ ]:
X = build_features(df, cfg.features)
y = extract_target(df)

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}, fraud rate: {y.mean():.2%}')
print(f'NaN values remaining: {X.isnull().sum().sum()}')
print(f'Category dtype columns: {(X.dtypes == "category").sum()}')
assert X.isnull().sum().sum() == 0, 'NaNs remain after build_features — check fill logic'
print('\nAll assertions passed.')